In [47]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [48]:
filename = "G:/neural_network_from_scratch/heart diseases/heart.csv"
df = pd.read_csv(filename)
df.head(1)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0


In [49]:
# training a model only on chestpaintype for heart disease
chest_df = df[['ChestPainType','HeartDisease']]

chest_pain_encoded = pd.get_dummies(
    chest_df['ChestPainType'],
    prefix='ChestPainType',
    dtype=int
)

chest_df = pd.concat([chest_df, chest_pain_encoded], axis=1)
chest_df.drop('ChestPainType', axis=1, inplace=True)

chest_df.head()

,HeartDisease,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA
0,0,0,1,0,0
1,1,0,0,1,0
2,0,0,1,0,0
3,1,1,0,0,0
4,0,0,0,1,0


In [50]:
from sklearn.model_selection import train_test_split

X = chest_df.drop('HeartDisease', axis=1).values
y = df['HeartDisease'].values.reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [51]:
X_train.shape

(734, 4)

In [52]:
y_train.shape

(734, 1)

In [53]:
# structure: 4 -> 8 -> 1

w1 = np.random.randn(4,8)*np.sqrt(2/4)  # 4 inputs and 8 neurons
b1 = np.zeros((1,8))

w2 = np.random.randn(8,1)*np.sqrt(2/8) # 8 inputs and 1 output
b2 = np.zeros((1,1))

def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    return 1 / (1 + np.exp(-np.clip(Z, -500, 500)))

def forward(X,w1,b1,w2,b2):
    z1 = np.dot(X,w1) + b1
    a1 = relu(z1)

    z2 = np.dot(a1,w2) + b2
    a2 = sigmoid(z2)

    return z1,a1,z2,a2

In [54]:
def loss_func(y,y_pred):

    epsilon = 1e-15
    y_pred = np.clip(y_pred,epsilon,1-epsilon)

    loss = -np.mean(
        y*np.log(y_pred) +
        (1-y)*np.log(1-y_pred)
    )
    return loss

def backward(X,y,z1,a1,a2,z2):
    m = X.shape[0]
    dz2 = a2 - y
    dw2 = np.dot(a1.T,dz2)/m
    db2 = np.sum(dz2)/m
    da1 = np.dot(dz2,w2.T)  
    dz1 = da1*(z1>0)
    dw1 = np.dot(X.T,dz1)/m
    db1 = np.sum(dz1)/m

    return dw1,db1,dw2,db2

def update_parameters(w1,b1,w2,b2,dw1,db1,dw2,db2,lr):
    w1 = w1 - lr*dw1
    b1 = b1 - lr*db1 
    w2 = w2 - lr*dw2
    b2 = b2 - lr*db2

    return w1,b1,w2,b2

In [55]:
learning_rate = 0.05
epochs = 5000

for epoch in range(epochs):

    z1,a1,z2,a2 = forward(X_train,w1,b1,w2,b2)
    loss = loss_func(y_train,a2)

    dw1,db1,dw2,db2 = backward(X_train,y_train,z1,a1,a2,z2)
    w1,b1,w2,b2 = update_parameters(w1,b1,w2,b2,dw1,db1,dw2,db2,learning_rate)
    if epoch % 500 == 0:
        print(f"Epoch: {epoch}, Loss: {loss:.6f}")

Epoch: 0, Loss: 0.643671
Epoch: 500, Loss: 0.555615
Epoch: 1000, Loss: 0.547916
Epoch: 1500, Loss: 0.545822
Epoch: 2000, Loss: 0.544950
Epoch: 2500, Loss: 0.544596
Epoch: 3000, Loss: 0.544466
Epoch: 3500, Loss: 0.544421
Epoch: 4000, Loss: 0.544406
Epoch: 4500, Loss: 0.544401


In [57]:
Z1, A1, Z2, y_pred = forward(
    X_val, w1, b1, w2, b2
)
predictions = (y_pred >= 0.5).astype(int)
accuracy = np.mean(predictions == y_val)
print("Test Accuracy:", accuracy * 100)

Test Accuracy: 80.43478260869566


In [58]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_val, predictions)
print(cm)

[[65 12]
 [24 83]]
